In [6]:
from __future__ import annotations
from typing import Annotated, List, Dict, Optional, Literal
from typing_extensions import TypedDict

from pathlib import Path
from pydantic import BaseModel, Field

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command

# LangChain
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, BaseMessage, SystemMessage
from langchain_core.tools import tool

# Tools
from langchain_tavily import TavilySearch
from langchain_community.document_loaders import WebBaseLoader
from langchain_experimental.utilities import PythonREPL

from dotenv import load_dotenv

load_dotenv()

# 1) 공용 상태 정의
class State(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    next: str

In [9]:
# 2) 리서치팀 도구
# Tavily 웹 검색: 상위 n개 결과를 반환
tavily_tool = TavilySearch(max_results=5)

@tool
def scrape_webpages(urls:List[str]):
    """요청한 웹 페이지 목록을 크롤링하여 본문 텍스트를 추출합니다."""
    loader = WebBaseLoader(
        urls,
        header_template={
            "User-Agent":(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
                "AppleWebKit/537.36 (KHTML,like Gecko)"
                "Chrome/122.0.0.0 Safari/537.36"
            )})
    docs = loader.load()
    return "\n\n".join(
        [
            f'<문서 제목="{doc.metadata.get("title","")}">\n{doc.page_content}\n</문서>'
            for doc in docs
        ]
    )


In [10]:
# 3) 문서팀 도구
WORKING_DIRECTORY = Path.cwd()

In [11]:
@tool
def create_outline(
    points: Annotated[List[str], "섹션/소제목 리스트"],
    file_name: Annotated[str, "개요를 저장할 파일 경로(상대/절대)"]):
    """개요(아웃라인)를 파일로 저장힙니다."""
    path = (WORKING_DIRECTORY / file_name)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for i, p in enumerate(points):
            f.write(f"{i + 1}. {p}\n")
    return f"개요 저장 완료: {path}"